# Train BEV Bike Classifier

Trains a lightweight CNN classifier on the hand-labeled BEV images from `label_clusters.ipynb`.

**Pipeline:**
1. Load labels from `data/labels/cluster_labels.csv`
2. Re-render BEV images for labeled clusters
3. Train an EfficientNet-B0 (pretrained on ImageNet) with 2-channel BEV input
4. Evaluate on a held-out validation split
5. Save the model to `data/models/bev_bike_classifier.pt`
6. Run inference on all unlabeled clusters to preview predictions

## Config

In [ ]:
from pathlib import Path

LABELS_FILE    = Path("data/labels/cluster_labels.csv")
MODELS_DIR     = Path("data/models")
MODEL_OUT      = MODELS_DIR / "bev_bike_classifier.pt"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

BEV_RESOLUTION  = 0.05
BEV_OUTPUT_SIZE = 64

VAL_FRACTION   = 0.2    # fraction held out for validation
EPOCHS         = 30
BATCH_SIZE     = 16
LR             = 3e-4
SEED           = 42

print(f"Labels file: {LABELS_FILE}")
print(f"Model out:   {MODEL_OUT}")

## Load labels

In [ ]:
import pandas as pd

df = pd.read_csv(LABELS_FILE)
print(f"Total labeled clusters: {len(df)}")
print(df["label"].value_counts().to_string())

# Group by source file so we can load each LAZ once
file_groups = df.groupby(["laz_file", "obstacles_file"])
print(f"\nSource file groups: {len(file_groups)}")

## Render BEV images for all labeled clusters

In [ ]:
import numpy as np
import laspy
import json
from shapely.geometry import shape

from utils.obstacle_extractor_2d import build_ground_grid, compute_heights
from utils.bev_renderer import extract_cluster_bev_from_arrays

X_list = []   # BEV images
y_list = []   # 0 = other, 1 = bike

for (laz_path, obs_path), group in file_groups:
    print(f"\nLoading {Path(laz_path).name} …")
    pc = laspy.read(laz_path)
    xyz = np.column_stack([
        np.asarray(pc.x, dtype=np.float64),
        np.asarray(pc.y, dtype=np.float64),
        np.asarray(pc.z, dtype=np.float64),
    ])
    labels_arr = (
        np.asarray(pc.label, dtype=np.int32)
        if "label" in pc.point_format.extra_dimension_names
        else np.zeros(len(xyz), dtype=np.int32)
    )
    grid, xmin, ymin = build_ground_grid(xyz, labels_arr)
    heights = compute_heights(xyz, grid, xmin, ymin)

    with open(obs_path) as f:
        gj = json.load(f)
    all_polygons = [shape(feat["geometry"]) for feat in gj["features"]]

    for _, row in group.iterrows():
        idx  = int(row["cluster_idx"])
        poly = all_polygons[idx]
        img  = extract_cluster_bev_from_arrays(
            xyz, heights, poly,
            resolution=BEV_RESOLUTION,
            output_size=BEV_OUTPUT_SIZE,
        )
        X_list.append(img)
        y_list.append(1 if row["label"] == "bike" else 0)

X = np.stack(X_list, axis=0)   # (N, H, W, 2)
y = np.array(y_list, dtype=np.int64)
print(f"\nDataset shape: {X.shape}")
print(f"Bike: {y.sum()}  |  Other: {(y==0).sum()}")

## Train / validation split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_FRACTION, stratify=y, random_state=SEED
)
print(f"Train: {len(X_train)}  |  Val: {len(X_val)}")

## Train EfficientNet-B0

We adapt the pretrained ImageNet weights by replacing the first conv layer to accept 2 input channels (height + density) instead of 3. The final classifier head is replaced with a binary output.

Install if needed: `pip install timm`

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import timm

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- Build model ---
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2)

# Replace stem conv: 3-channel → 2-channel (average the first two channels of pretrained weights)
old_conv = model.conv_stem
new_conv = nn.Conv2d(
    2, old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=old_conv.bias is not None,
)
with torch.no_grad():
    new_conv.weight.copy_(old_conv.weight[:, :2, :, :])
model.conv_stem = new_conv
model = model.to(device)

# --- Datasets ---
# BEV images are (H, W, 2); PyTorch expects (N, C, H, W)
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32).permute(0, 3, 1, 2)

train_ds = TensorDataset(to_tensor(X_train), torch.tensor(y_train))
val_ds   = TensorDataset(to_tensor(X_val),   torch.tensor(y_val))
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

# Class weights to handle bike/other imbalance
n_bike  = int(y_train.sum())
n_other = len(y_train) - n_bike
weight  = torch.tensor([n_bike / len(y_train), n_other / len(y_train)],
                        dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weight)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# --- Training loop ---
best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)
    scheduler.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(1)
            correct += (preds == yb).sum().item()
            total   += len(yb)
    val_acc = correct / total if total else 0.0

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_OUT)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  "
              f"loss={train_loss/len(train_ds):.4f}  "
              f"val_acc={val_acc:.3f}  (best={best_val_acc:.3f})")

print(f"\nBest val accuracy: {best_val_acc:.3f}")
print(f"Model saved → {MODEL_OUT}")

## Evaluate: confusion matrix and per-class metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model.load_state_dict(torch.load(MODEL_OUT, map_location=device))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in val_dl:
        xb = xb.to(device)
        all_preds.extend(model(xb).argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())

print(classification_report(all_true, all_preds, target_names=["other", "bike"]))

cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=["other", "bike"],
            yticklabels=["other", "bike"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Validation confusion matrix")
plt.tight_layout()
plt.show()

## Preview: show predicted bike clusters

Runs the model on all clusters (labeled + unlabeled) from a single tile and shows the ones predicted as bikes.

In [ ]:
PREVIEW_LAZ      = Path("/Users/minkeverweij/UPSW_neo/data/output/labeled_pointcloud/bgt_labeled_120300_489300.laz")
PREVIEW_OBSTACLES = Path("/Users/minkeverweij/UPSW_neo/data/output/obstacles/obstacles_2d_120300_488900.geojson")
SCORE_THRESHOLD  = 0.6   # confidence threshold for 'bike'

print("Loading …")
pc2 = laspy.read(str(PREVIEW_LAZ))
xyz2 = np.column_stack([
    np.asarray(pc2.x, dtype=np.float64),
    np.asarray(pc2.y, dtype=np.float64),
    np.asarray(pc2.z, dtype=np.float64),
])
labels2 = (
    np.asarray(pc2.label, dtype=np.int32)
    if "label" in pc2.point_format.extra_dimension_names
    else np.zeros(len(xyz2), dtype=np.int32)
)
grid2, xmin2, ymin2 = build_ground_grid(xyz2, labels2)
heights2 = compute_heights(xyz2, grid2, xmin2, ymin2)

with open(PREVIEW_OBSTACLES) as f:
    gj2 = json.load(f)
polys2 = [shape(feat["geometry"]) for feat in gj2["features"]]

print("Rendering and scoring …")
all_imgs, all_scores = [], []
model.eval()
with torch.no_grad():
    for poly in polys2:
        img = extract_cluster_bev_from_arrays(
            xyz2, heights2, poly,
            resolution=BEV_RESOLUTION, output_size=BEV_OUTPUT_SIZE
        )
        t = to_tensor(img[np.newaxis]).to(device)     # (1, 2, H, W)
        prob = torch.softmax(model(t), dim=1)[0, 1].item()
        all_imgs.append(img)
        all_scores.append(prob)

bike_idx = [i for i, s in enumerate(all_scores) if s >= SCORE_THRESHOLD]
print(f"Predicted bikes (score ≥ {SCORE_THRESHOLD}): {len(bike_idx)} / {len(polys2)}")

# Show predicted bike clusters
import matplotlib.pyplot as plt
ncols = 8
nrows = max(1, (len(bike_idx) + ncols - 1) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.4, nrows * 1.4), squeeze=False)
for j, ax in enumerate(axes.flat):
    if j >= len(bike_idx):
        ax.axis("off"); continue
    i = bike_idx[j]
    img = all_imgs[i]
    rgb = np.zeros((*img.shape[:2], 3), dtype=np.float32)
    rgb[..., 1] = img[..., 0]
    rgb[..., 2] = img[..., 1]
    ax.imshow(rgb, origin="lower", interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{i}\n{all_scores[i]:.2f}", fontsize=7, pad=2)
plt.suptitle(f"Predicted bike clusters (threshold={SCORE_THRESHOLD})", fontsize=9)
plt.tight_layout()
plt.show()